# 03 Validation

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/Gates Foundation/Building Dataset Validation/")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
URBAN_VALIDATION_DEBUG_LOG = "/content/drive/MyDrive/Gates Foundation/Building Dataset Validation/outputs/logs/ssd_juba_globfp_2026-03-28_15-10-25.log"
os.environ["URBAN_VALIDATION_DEBUG_LOG"] = URBAN_VALIDATION_DEBUG_LOG

In [3]:
# ! pip install psutil

In [4]:
import os
os.environ["SJOIN_CHUNK_SIZE"] = "5000"
os.environ["AREA_CHUNK_SIZE"]  = "5000"

In [5]:
from __future__ import annotations
import gc
import logging
import sys
from pathlib import Path
 
import pandas as pd
import yaml
 
from src.validator import Validator
from src.output import load_config
 

In [6]:
import importlib
import src.metrics, src.utils, src.validator
importlib.reload(src.metrics)
importlib.reload(src.utils)
importlib.reload(src.validator)

<module 'src.validator' from '/content/drive/.shortcut-targets-by-id/1xTSKkjN4wft_0kljbwaAGKjTGEBsHZm1/Gates Foundation/Building Dataset Validation/src/validator.py'>

In [7]:
def _build_logger(name: str = "Urban_Validator") -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        fmt = logging.Formatter(
            "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
            datefmt="%Y-%m-%d %H:%M:%S",
        )
        sh = logging.StreamHandler(sys.stdout)
        sh.setFormatter(fmt)
        logger.addHandler(sh)
    logger.setLevel(logging.INFO)
    return logger
 

In [8]:
def load_tracker(tracker_path: str | Path) -> pd.DataFrame:
    """Load the AOI tracker CSV with normalised column names and stripped strings."""
    df = pd.read_csv(tracker_path, dtype=str)
    df.columns = df.columns.str.strip()
    df = df.apply(lambda col: col.str.strip() if col.dtype == object else col)
    return df


def _is_truthy(val: str) -> bool:
    return str(val).strip().lower() in ("true", "yes", "1")


def resolve_sub_areas(
    group: pd.DataFrame,
    data_dir: Path,
    logger: logging.Logger,
) -> list[dict]:
    """
    Turn a group of CSV rows (all sharing the same Dataset code) into a list
    of validated sub-area dicts ready for Validator.

    Each returned dict has:
        sub_area_id : str   – derived from aoi_file_name stem
        aoi         : Path  – resolved, confirmed to exist
        reference   : Path  – resolved, confirmed to exist
    """
    sub_areas = []

    for _, row in group.iterrows():
        city         = row["Dataset code"]
        folder       = row["dataset_folder_name"]
        # CSV empty cells arrive as float NaN even with dtype=str on some pandas
        # versions — coerce every field to str and strip whitespace defensively.
        aoi_file = "" if pd.isna(row.get("aoi_file_name")) else str(row.get("aoi_file_name", "")).strip()
        ref_file = "" if pd.isna(row.get("reference_file_name")) else str(row.get("reference_file_name", "")).strip()
        has_aoi  = _is_truthy(row.get("has_aoi_file", "false"))
        has_ref  = _is_truthy(row.get("has_reference_file", "false"))

        # Skip rows that are known incomplete
        if not has_aoi:
            logger.warning("  Skipping row (no AOI file): %s / %s", city, ref_file or "?")
            continue
        if not has_ref:
            logger.warning("  Skipping row (no reference file): %s / %s", city, aoi_file or "?")
            continue
        if not aoi_file:
            logger.warning("  Skipping row (empty aoi_file_name): %s", city)
            continue
        if not ref_file:
            logger.warning("  Skipping row (empty reference_file_name): %s", city)
            continue

        # Build paths from the convention
        aoi_path = data_dir / folder / "aoi"    / aoi_file
        ref_path = data_dir / folder / "vector" / ref_file

        # Hard-check existence so problems surface immediately
        if not aoi_path.exists():
            logger.error("  AOI file not found, skipping: %s", aoi_path)
            continue
        if not ref_path.exists():
            logger.error("  Reference file not found, skipping: %s", ref_path)
            continue

        # Use the AOI filename stem as the sub-area identifier
        # e.g. "jubacitycenter_aoi.geojson" → "jubacitycenter"
        sub_area_id = Path(aoi_file).stem.replace("_aoi", "")

        sub_areas.append({
            "sub_area_id": sub_area_id,
            "aoi":         aoi_path,
            "reference":   ref_path,
        })

    return sub_areas


def _process_sub_area(
    sub_area: dict,
    city: str,
    cfg: dict,
    logger: logging.Logger,
) -> tuple[str, str, pd.DataFrame | None]:
    """
    Process a single sub-area in an isolated scope for better memory cleanup.
    
    Returns tuple of (city, sub_area_id, summary_df or None if failed).
    """
    import psutil
    
    sub_area_id = sub_area["sub_area_id"]
    
    # Monitor memory before processing
    mem_before = psutil.virtual_memory()
    logger.info(
        "    Sub-area: %s [RAM: %.1f / %.1f GB (%.0f%%)]",
        sub_area_id,
        mem_before.used / 1e9, mem_before.total / 1e9, mem_before.percent
    )

    v = None
    summary = None

    try:
        v = Validator(
            cfg,
            city=city,
            aoi=sub_area["aoi"],
            reference=sub_area["reference"],
            log=logger,
        )

        metrics_all, matches_all = v.compute_vector_metrics()

        if metrics_all.empty:
            logger.warning("%s / %s: no metrics produced, skipping.", city, sub_area_id)
            return (city, sub_area_id, None)

        # Delete large objects before visualization
        del metrics_all, matches_all
        for _ in range(3):
            gc.collect()

        summary = v.visualize_metrics()
        logger.info("%s / %s: done ✓", city, sub_area_id)
        return (city, sub_area_id, summary)

    except Exception as e:
        logger.exception("Error processing '%s / %s': %s", city, sub_area_id, e)
        return (city, sub_area_id, None)

    finally:
        # Cleanup local scope variables
        try: del v
        except NameError: pass
        try: del summary
        except NameError: pass
        
        # Force cleanup before returning
        for _ in range(5):
            gc.collect()
        
        mem_after = psutil.virtual_memory()
        logger.info(
            "    Cleanup [RAM: %.1f / %.1f GB (%.0f%%)]",
            mem_after.used / 1e9, mem_after.total / 1e9, mem_after.percent
        )


In [ ]:
vvasr



















In [ ]:
import psutil, threading, time
 
def _monitor():
    while True:
        m = psutil.virtual_memory()
        print(
            f"\r[RAM] {m.used/1e9:.1f}/{m.total/1e9:.1f} GB  "
            f"({m.percent:.0f}%)  {time.strftime('%H:%M:%S')}",
            end="", flush=True,
        )
        time.sleep(10)
 
threading.Thread(target=_monitor, daemon=True).start()

[RAM] 0.9/54.8 GB  (3%)  03:25:38

: 

In [11]:
config_path = Path(
    "/content/drive/MyDrive/Gates Foundation/Building Dataset Validation/"
    "configs/validation_configs.yaml"
)

# OPTION 1: Run all cities and sub-areas (may use lots of memory)
# run_all(config_path=config_path, quality_filter=True)

# OPTION 2: Process specific cities one at a time (safer for memory)
# This allows you to run multiple times, processing a few cities each session
# Uncomment one and modify the city names as needed:
# run_all(config_path=config_path, cities=["juba"], quality_filter=True)
# run_all(config_path=config_path, cities=["curacao"], quality_filter=True)
# run_all(config_path=config_path, cities=["nairobi"], quality_filter=True)

# OPTION 3: Run high-quality cities only (recommended for limited memory)
run_all(config_path=config_path, quality_filter=True)


2026-03-29 03:25:38 | INFO | Urban_Validator | Loaded tracker: 59 rows, 57 unique Dataset codes
INFO:Urban_Validator:Loaded tracker: 59 rows, 57 unique Dataset codes
2026-03-29 03:25:38 | INFO | Urban_Validator | Quality filter applied: 59 rows remain
INFO:Urban_Validator:Quality filter applied: 59 rows remain
2026-03-29 03:25:38 | INFO | Urban_Validator | Processing 57 cities.
INFO:Urban_Validator:Processing 57 cities.
2026-03-29 03:25:38 | INFO | Urban_Validator | ============================================================
INFO:Urban_Validator:============================================================
2026-03-29 03:25:38 | INFO | Urban_Validator | City 1 / 57: ant-curacao
INFO:Urban_Validator:City 1 / 57: ant-curacao
2026-03-29 03:25:38 | INFO | Urban_Validator | ============================================================
INFO:Urban_Validator:============================================================
2026-03-29 03:25:38 | INFO | Urban_Validator | ant-curacao: found 1 sub-area(s

[RAM] 1.1/54.8 GB  (3%)  03:25:48

2026-03-29 03:25:48 | INFO | Urban_Validator | All figures saved → /content/drive/MyDrive/Gates Foundation/Building Dataset Validation/outputs/figures/blz-burrell-boom
INFO:Urban_Validator:All figures saved → /content/drive/MyDrive/Gates Foundation/Building Dataset Validation/outputs/figures/blz-burrell-boom
2026-03-29 03:25:48 | INFO | Urban_Validator | blz-burrell-boom / burrell-boom: done ✓
INFO:Urban_Validator:blz-burrell-boom / burrell-boom: done ✓
2026-03-29 03:25:49 | INFO | Urban_Validator |     Cleanup [RAM: 1.1 / 54.8 GB (3%)]
INFO:Urban_Validator:    Cleanup [RAM: 1.1 / 54.8 GB (3%)]
2026-03-29 03:25:49 | INFO | Urban_Validator | ============================================================
INFO:Urban_Validator:============================================================
2026-03-29 03:25:49 | INFO | Urban_Validator | City 3 / 57: bra-nova-sussuarana
INFO:Urban_Validator:City 3 / 57: bra-nova-sussuarana
2026-03-29 03:25:49 | INFO | Urban_Validator | ===========================

[RAM] 1.3/54.8 GB  (3%)  03:25:58

2026-03-29 03:26:00 | INFO | Urban_Validator | All figures saved → /content/drive/MyDrive/Gates Foundation/Building Dataset Validation/outputs/figures/col-san-antonio-de-prado
INFO:Urban_Validator:All figures saved → /content/drive/MyDrive/Gates Foundation/Building Dataset Validation/outputs/figures/col-san-antonio-de-prado
2026-03-29 03:26:00 | INFO | Urban_Validator | col-san-antonio-de-prado / san-antonio-de-prado: done ✓
INFO:Urban_Validator:col-san-antonio-de-prado / san-antonio-de-prado: done ✓
2026-03-29 03:26:01 | INFO | Urban_Validator |     Cleanup [RAM: 1.3 / 54.8 GB (3%)]
INFO:Urban_Validator:    Cleanup [RAM: 1.3 / 54.8 GB (3%)]
2026-03-29 03:26:01 | INFO | Urban_Validator | ============================================================
INFO:Urban_Validator:============================================================
2026-03-29 03:26:01 | INFO | Urban_Validator | City 5 / 57: cvg-saint vincent grenadines
INFO:Urban_Validator:City 5 / 57: cvg-saint vincent grenadines
2026-03-

[RAM] 1.4/54.8 GB  (4%)  03:26:08

2026-03-29 03:26:11 | INFO | Urban_Validator | All figures saved → /content/drive/MyDrive/Gates Foundation/Building Dataset Validation/outputs/figures/gha-aiyim-sraha
INFO:Urban_Validator:All figures saved → /content/drive/MyDrive/Gates Foundation/Building Dataset Validation/outputs/figures/gha-aiyim-sraha
2026-03-29 03:26:11 | INFO | Urban_Validator | gha-aiyim-sraha / aiyim-sraha: done ✓
INFO:Urban_Validator:gha-aiyim-sraha / aiyim-sraha: done ✓
2026-03-29 03:26:12 | INFO | Urban_Validator |     Cleanup [RAM: 1.4 / 54.8 GB (4%)]
INFO:Urban_Validator:    Cleanup [RAM: 1.4 / 54.8 GB (4%)]
2026-03-29 03:26:12 | INFO | Urban_Validator | ============================================================
INFO:Urban_Validator:============================================================
2026-03-29 03:26:12 | INFO | Urban_Validator | City 8 / 57: gha-dansoman
INFO:Urban_Validator:City 8 / 57: gha-dansoman
2026-03-29 03:26:12 | INFO | Urban_Validator | ===============================================

[RAM] 49.5/54.8 GB  (92%)  03:29:58

: 

: 